# Formulaire — Probabilistic Modeling & Bayesian Reasoning

> **Mode d'emploi** : remplace uniquement les variables dans les blocs `# === VARIABLES À CHANGER ===`. Le reste du code tourne tel quel.

---

## EXERCICE 1 — Simulation & variabilité

### Formules clés

| Grandeur | Formule |
|---|---|
| Fréquence empirique | `freq = data.mean()` |
| Écart-type théorique | $\sigma = \sqrt{\dfrac{p(1-p)}{n}}$ |
| Biais de l'estimateur | $\mathbb{E}[\hat{p}] = p$ → **sans biais** |

### Ce qui change d'un exam à l'autre
- `p_anomaly` : probabilité de l'événement rare
- `n` : taille d'un échantillon
- `n_simulations` : nombre de répétitions (souvent 2500)
- Les tailles à comparer dans Q2 (ex: 50 et 800)

### Interprétation automatique
- Std **grande** → estimateur peu fiable, échantillon petit
- Std **petite** → estimateur précis, grand n
- Moyenne des simulations ≈ p_réel → estimateur **sans biais**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === VARIABLES À CHANGER ===
SEED        = 52
P           = 0.10
N           = 190
N_SIMUL     = 3000
N_COMPARE   = [20, 500]
# ======= CODE PROF =========
np.random.seed(SEED)
data = np.random.binomial(1, P, size=N)
# ===========================

freq_echan_uni = data.mean().round(4)
print("Fréquence empirique : ", freq_echan_uni)

list_echan = []
for x in range(N_SIMUL):
    echan = np.random.binomial(1, P, size=N)
    list_echan.append(echan.mean())

list_echan = np.array(list_echan)
moy_list_echan = list_echan.mean()

plt.hist(list_echan, bins=40)
plt.title(f"Distribution de {N_SIMUL} échantillons de taille n={N}")
plt.axvline(moy_list_echan, color="red",    label="moy_list_echan")
plt.axvline(P,              color="orange", label="vraie_valeur_p")
plt.axvline(freq_echan_uni, color="pink",   label="freq_echan_uni")
plt.legend()
plt.show()

# Q2 — tailles alternatives
for n_test in N_COMPARE:
    list_echan2 = []
    for x in range(N_SIMUL):
        echan = np.random.binomial(1, P, size=n_test)
        list_echan2.append(echan.mean())
    list_echan2 = np.array(list_echan2)
    print(f"n={n_test} → mean={list_echan2.mean():.4f}, std={list_echan2.std():.4f}")
    plt.hist(list_echan2, bins=40)
    plt.title(f"Distribution de {N_SIMUL} échantillons de taille n={n_test}")
    plt.axvline(list_echan2.mean(), color="red",    label="moyenne")
    plt.axvline(P,                  color="orange", label="vraie_valeur_p")
    plt.legend()
    plt.show()

---
## EXERCICE 2 — Hypothèses discrètes

### Formules clés

$$P(\text{alerte} \mid H) = p_{\text{intrusion}} \times \text{detect\_rate}_H + (1 - p_{\text{intrusion}}) \times p_{\text{faux positif}}$$

**Bayes discret (prior 50/50) :**
$$P(H=1 \mid \text{data}) = \frac{\mathcal{L}(H=1)}{\mathcal{L}(H=0) + \mathcal{L}(H=1)}$$

où $\mathcal{L}(H) = \text{Binomial}(k \mid n, p_{\text{alerte} \mid H})$

### Ce qui change d'un exam à l'autre
- `P_INTRUSION` : probabilité d'une vraie intrusion
- `DETECT_H0` / `DETECT_H1` : taux de détection selon l'état du système
- `P_FAUX_POS` : taux de faux positifs
- `N` : nombre d'observations

### Interprétation
- P(H=1|data) **proche de 0.5** → les deux hypothèses sont indiscernables (taux d'alertes proches)
- P(H=1|data) **proche de 0 ou 1** → les données permettent de trancher
- La différence `p_alert_H0 - p_alert_H1` quantifie la **discriminabilité** du test

In [ ]:
import numpy as np
import pymc as pm
import arviz as az

# === VARIABLES À CHANGER ===
SEED           = 13
N              = 280
P_FAILURE      = 0.07
DETECT_IF_FAIL = 0.89
DETECT_IF_OK   = 0.10
# ======= CODE PROF =========

np.random.seed(SEED)
failure = np.random.binomial(1, P_FAILURE, size=N)

p_alert_H0 = DETECT_IF_OK
p_alert_H1 = DETECT_IF_FAIL

alerts = np.where(
    failure == 1,
    np.random.binomial(1, DETECT_IF_FAIL, size=N),
    np.random.binomial(1, DETECT_IF_OK,   size=N)
)

# ==========================

n_alerts = alerts.sum()

with pm.Model() as model:

    H = pm.Bernoulli("H", p=0.5)

    likelihood = pm.math.switch(pm.math.eq(H, 1), p_alert_H1, p_alert_H0)

    obs = pm.Binomial("obs", n=N, p=likelihood, observed=n_alerts)

    idata = pm.sample(2000, tune=1000, idata_kwargs={"log_likelihood": True})

print(az.summary(idata, var_names=["H"]))
print(f"\nP(H=1 | données) = {float(idata.posterior['H'].mean()):.4f}")

---
## EXERCICE 3 — Inférence Bayésienne & décision

### Formules clés

**Mise à jour conjuguée Beta-Binomiale :**
$$\text{Prior } \text{Beta}(\alpha, \beta) \;+\; k \text{ succès sur } n \;\Rightarrow\; \text{Postérieur } \text{Beta}(\alpha+k,\; \beta+n-k)$$

| Grandeur | Formule |
|---|---|
| Moyenne a posteriori | $\mu = \dfrac{\alpha_{\text{post}}}{\alpha_{\text{post}} + \beta_{\text{post}}}$ |
| Influence du prior | $\dfrac{\alpha+\beta}{\alpha+\beta+n}$ |
| Prior non-informatif | $\text{Beta}(1, 1)$ |

### Priors courants

| Prior | Signification |
|---|---|
| Beta(1, 1) | Non-informatif (uniforme) |
| Beta(2, 2) | Légèrement centré sur 0.5 |
| Beta(6, 2) | Optimiste (p élevé probable) |
| Beta(2, 6) | Pessimiste (p faible probable) |

### Ce qui change d'un exam à l'autre
- `N_OBS` : nombre d'observations
- `K_SUCCESS` : nombre de succès
- `ALPHA_PRIOR` / `BETA_PRIOR` : paramètres du prior
- `ALPHA_OPT` / `BETA_OPT` : prior alternatif (Q2)
- `SEUIL_*` : seuils de la règle de décision

In [ ]:
import numpy as np
import pymc as pm
import arviz as az

# === VARIABLES À CHANGER ===
SEED         = 18

N_USERS      = 40
SUCCESS      = 27

ALPHA_PRIOR  = 1      # prior non-informatif : Beta(1,1)
BETA_PRIOR   = 1

ALPHA_ALT    = 2      # prior alternatif (Q2)
BETA_ALT     = 8

SEUIL_CALCUL = 0.62   # seuil pour P(p > x)
# ===========================

np.random.seed(SEED)

# prior non-informatif
with pm.Model() as model_bayesian:
    p   = pm.Beta("p", alpha=ALPHA_PRIOR, beta=BETA_PRIOR)
    obs = pm.Binomial("obs", p=p, n=N_USERS, observed=SUCCESS)
    idata = pm.sample(2000, tune=1000)

print(az.summary(idata, var_names=["p"]))

samples = idata.posterior["p"].values.flatten()
proba = (samples > SEUIL_CALCUL).mean()
print(f"P(p > {SEUIL_CALCUL}) = {proba}")

In [ ]:
# prior alternatif (Q2)
with pm.Model() as model_bayesian:
    p   = pm.Beta("p", alpha=ALPHA_ALT, beta=BETA_ALT)
    obs = pm.Binomial("obs", p=p, n=N_USERS, observed=SUCCESS)
    idata = pm.sample(2000, tune=1000)

print(az.summary(idata, var_names=["p"]))

samples = idata.posterior["p"].values.flatten()
proba = (samples > SEUIL_CALCUL).mean()
print(f"P(p > {SEUIL_CALCUL}) = {proba}")

---
## EXERCICE 4 — Comparaison de modèles & PPC

### Formules clés

**Moyenne de référence pour M1 :**
$$\mu_{M1} = \frac{1}{n}\sum x_i$$

**LOO (Leave-One-Out cross-validation) :**
- Score **plus élevé** (moins négatif) = modèle **meilleur**
- `elpd_diff` : différence de score — si grande et `dse` petite → différence significative

**Posterior Predictive Check (PPC) :**
- Simule de nouvelles données depuis le postérieur
- Un bon modèle reproduit la **forme** des vraies données

### Ce qui change d'un exam à l'autre
- `DATA` : tableau des observations
- `GROUP` : tableau 0/1 indiquant le régime de chaque obs
- Priors sur `mu`, `mu0`, `mu1`, `sigma` (à adapter à l'échelle des données)

### Règle rapide pour choisir les priors
- Centrer `mu` sur la moyenne globale des données
- Centrer `mu0` / `mu1` sur la moyenne de chaque groupe visible
- `sigma` : HalfNormal avec sigma ≈ écart-type intra-groupe attendu

In [ ]:
import numpy as np
import pymc as pm
import arviz as az

# === VARIABLES À CHANGER ===
DATA  = np.array([72, 74, 71, 73, 45, 43, 46, 44, 72, 44])
GROUP = np.array([0,   0,  0,  0,  1,  1,  1,  1,  0,  1])

MU_M1      = 58    # moyenne prior M1 ≈ moyenne globale des données
SIGMA_M1   = 14    # std prior M1 ≈ std globale des données
MU_G0      = 72    # moyenne prior groupe 0
MU_G1      = 44    # moyenne prior groupe 1
SIGMA_M2   = 2     # std prior des moyennes M2
SIGMA_OBS  = 4     # std des observations M2
# ===========================

print(data.mean())
print(data.std())

print("\nGroupe 0 :\n")
group0 = DATA[GROUP == 0]
print(f"Moyenne group0 = {group0.mean()}")
print(f"Std group0 = {group0.std()}")

print("\nGroupe 1 :\n")
group1 = DATA[GROUP == 1]
print(f"Moyenne group1 = {group1.mean()}")
print(f"Std group1 = {group1.std()}")

# modèle simple
with pm.Model() as model_single:
    mu = pm.Normal("mu", mu=MU_M1, sigma=SIGMA_M1)
    y  = pm.Normal("y",  mu=mu, sigma=SIGMA_M1*2, observed=DATA)
    idata_single = pm.sample(2000, tune=1000, idata_kwargs={"log_likelihood": True})

print(az.summary(idata_single, var_names=["mu"]))

In [ ]:
# modèle double
with pm.Model() as model_double:
    mu = pm.Normal("mu", mu=[MU_G0, MU_G1], sigma=SIGMA_M2, shape=2)
    y  = pm.Normal("y",  mu=mu[GROUP], sigma=SIGMA_OBS, observed=DATA)
    idata_double = pm.sample(2000, tune=1000, idata_kwargs={"log_likelihood": True})

print(az.summary(idata_double, var_names=["mu"]))

In [ ]:
# comparaison
comparison = az.compare({"Modèle simple": idata_single, "Modèle double": idata_double})
print(comparison)

In [ ]:
# ppc
with model_single:
    ppc_single = pm.sample_posterior_predictive(idata_single)
az.plot_ppc(ppc_single)

with model_double:
    ppc_double = pm.sample_posterior_predictive(idata_double)
az.plot_ppc(ppc_double)

---
## Aide-mémoire rapide

### Règles d'interprétation universelles

| Situation | Ce qu'il faut dire |
|---|---|
| Std simulée ≈ théorique | Simulation correcte |
| Std diminue quand n augmente | Loi des grands nombres |
| P(H=1\|data) ≈ 0.5 | Hypothèses indiscernables (taux proches) |
| P(H=1\|data) ≈ 0 ou 1 | Données discriminantes |
| Prior influence ++ | n petit ou prior informatif |
| LOO M2 > LOO M1 | M2 prédit mieux, mais vérifier PPC |
| PPC concentré sur les données | Modèle crédible |

### Formule Beta conjuguée (à mémoriser)

$$\boxed{\text{Beta}(\alpha, \beta) + (k \text{ succès}, n{-}k \text{ échecs}) \;\Rightarrow\; \text{Beta}(\alpha+k,\; \beta+n-k)}$$

### Influence du prior

$$\boxed{\text{Influence prior} = \frac{\alpha+\beta}{\alpha+\beta+n}}$$

Plus $n$ est grand, plus cette fraction tend vers 0 → les données prennent le dessus.